# Entanglement protocols

Five textbook protocols, each run on the twins with and without device noise: Deutsch–Jozsa, superdense coding, a CHSH
Bell test, entanglement swapping with feed-forward, and Simon's algorithm.

In [1]:
from qputwin_demo import *

connect()

QPUTwin at http://localhost:8000: ready, 7 twins reachable


{'status': 'ready',
 'instance': 'c88add07efb8:7:a77a42',
 'job_store': 'ok',
 'backends_reachable': ['quantinuum_helios',
  'ionq_forte',
  'rigetti_ankaa_3',
  'iqm_emerald',
  'ibm_heron',
  'ibm_nighthawk',
  'google_willow']}

## 1. Deutsch–Jozsa

One query decides whether a function on three bits is constant or balanced: the input register reads 000 only for a constant function.

In [2]:
def deutsch_jozsa(kind):
    oracle = {"constant 0": [], "constant 1": ["x q[3]"],
              "balanced (parity)": ["cx q[0],q[3]", "cx q[1],q[3]", "cx q[2],q[3]"],
              "balanced (first bit)": ["cx q[0],q[3]"]}[kind]
    lines = ["x q[3]", "h q[0]", "h q[1]", "h q[2]", "h q[3]", *oracle, "h q[0]", "h q[1]", "h q[2]"]
    return program(4, lines, measure=False) + "".join(f"measure q[{i}] -> c[{i}];\n" for i in range(3))

rows = []
for kind in ("constant 0", "constant 1", "balanced (parity)", "balanced (first bit)"):
    row = {"oracle": kind}
    for b in ("quantinuum_helios", "iqm_emerald", "google_willow"):
        r = run(b, deutsch_jozsa(kind), shots=4000, noise=True, routing=True)
        row[f"P(000) {NAMES[b]}"] = round(sum(v for k, v in r["result"]["counts"].items() if k[-3:] == "000") / 4000, 3)
    rows.append(row)
pd.DataFrame(rows).set_index("oracle")

,P(000) Helios,P(000) Emerald,P(000) Willow
oracle,,,
constant 0,0.999,0.911,0.993
constant 1,0.999,0.911,0.993
balanced (parity),0.000,0.001,0.000
balanced (first bit),0.001,0.031,0.009


## 2. Superdense coding

Two classical bits travel on one qubit of a Bell pair: the sender applies I, X, Z or ZX, and the receiver decodes with CX and H.

In [3]:
# bits b1 b0: Z on the sender's qubit carries b0, X carries b1; the decoder returns them as q1 q0
encodings = {"00": [], "10": ["x q[0]"], "01": ["z q[0]"], "11": ["x q[0]", "z q[0]"]}
rows = []
for message, ops in encodings.items():
    prog = program(2, ["h q[0]", "cx q[0],q[1]", *ops, "cx q[0],q[1]", "h q[0]"])
    row = {"sent (b1 b0)": message}
    for b in ("quantinuum_helios", "ionq_forte", "ibm_heron"):
        counts = run(b, prog, shots=4000, noise=True, routing=True)["result"]["counts"]
        row[NAMES[b]] = f"{max(counts, key=counts.get)} ({counts.get(max(counts, key=counts.get)) / 4000:.3f})"
    rows.append(row)
pd.DataFrame(rows).set_index("sent (b1 b0)")

,Helios,Forte,Heron r3
sent (b1 b0),,,
00,00 (0.999),00 (0.981),00 (1.000)
10,10 (0.999),10 (0.983),10 (0.997)
01,01 (0.999),01 (0.980),01 (0.998)
11,11 (0.999),11 (0.981),11 (0.991)


Each cell shows the most frequent received message (q1 q0) and how often it arrived.

## 3. A CHSH Bell test

Measuring a Bell pair along angles a, a′ and b, b′ gives S = E(a,b) + E(a,b′) + E(a′,b) − E(a′,b′). Local hidden-variable theories allow |S| ≤ 2; quantum mechanics reaches 2√2 ≈ 2.83. Noise pulls S down toward 2 and below.

In [4]:
def correlation(backend, theta_a, theta_b, noise, seed):
    prog = program(2, ["h q[0]", "cx q[0],q[1]", f"ry({-theta_a}) q[0]", f"ry({-theta_b}) q[1]"])
    c = run(backend, prog, shots=20000, noise=noise, routing=True, seed=seed)["result"]["counts"]
    return (c.get("00", 0) + c.get("11", 0) - c.get("01", 0) - c.get("10", 0)) / 20000

a, a2, b0, b2 = 0, np.pi / 2, np.pi / 4, -np.pi / 4
rows = []
for b in ALL_BACKENDS:
    for noise in (None, True):
        # a different seed per setting: one shared seed would correlate the four sampling errors
        S = (correlation(b, a, b0, noise, 11) + correlation(b, a, b2, noise, 12)
             + correlation(b, a2, b0, noise, 13) - correlation(b, a2, b2, noise, 14))
        rows.append({"twin": NAMES[b], "noise": "device" if noise else "ideal", "S": round(S, 3)})
t = pd.DataFrame(rows).pivot(index="twin", columns="noise", values="S")
t["violates |S| <= 2"] = t["device"] > 2
t

noise,device,ideal,violates |S| <= 2
twin,,,
Ankaa-3,2.439,2.816,True
Emerald,2.469,2.816,True
Forte,2.745,2.816,True
Helios,2.808,2.816,True
Heron r3,2.789,2.816,True
Nighthawk,2.714,2.816,True
Willow,2.741,2.816,True


## 4. Entanglement swapping

Two Bell pairs (0,1) and (2,3). A Bell measurement on qubits 1 and 2, with feed-forward corrections on qubit 3, entangles qubits 0 and 3, which never interacted. Their parity agrees in both the Z and the X basis.

In [5]:
def swapping(basis):
    rotate = "h q[0];\nh q[3];\n" if basis == "X" else ""
    return f"""OPENQASM 3;
include "stdgates.inc";
qubit[4] q;
bit[2] m;
bit[2] out;
h q[0]; cx q[0], q[1];
h q[2]; cx q[2], q[3];
cx q[1], q[2]; h q[1];
m[0] = measure q[1];
m[1] = measure q[2];
if (m[1] == 1) {{ x q[3]; }}
if (m[0] == 1) {{ z q[3]; }}
{rotate}out[0] = measure q[0];
out[1] = measure q[3];
"""

for b in ("quantinuum_helios", "ionq_forte"):
    for basis in ("Z", "X"):
        c = run(b, swapping(basis), shots=8000, noise=True)["result"]["counts"]
        # keys list every classical bit, highest first: out[1] out[0] m[1] m[0] (see result["cregs"])
        agree = sum(v for k, v in c.items() if k[0] == k[1]) / 8000
        print(f"{NAMES[b]:7s} {basis} basis: qubits 0 and 3 agree in {agree:.3f} of shots")

Helios  Z basis: qubits 0 and 3 agree in 0.998 of shots
Helios  X basis: qubits 0 and 3 agree in 0.998 of shots
Forte   Z basis: qubits 0 and 3 agree in 0.976 of shots
Forte   X basis: qubits 0 and 3 agree in 0.976 of shots


Without the swap, qubits 0 and 3 would agree half the time in either basis. Agreement near 1 in both bases means they share a Bell state.

## 5. Simon's algorithm

A two-to-one function f(x) = f(x ⊕ s) with a hidden two-bit string s. Every measured y satisfies y · s = 0 (mod 2), so a few samples reveal s.

In [6]:
def simon(secret):
    # copy x into the output register, then XOR x0 into the output wherever the secret has a 1
    lines = ["h q[0]", "h q[1]", "cx q[0],q[2]", "cx q[1],q[3]"]
    lines += [f"cx q[0],q[{2 + i}]" for i, bit in enumerate(reversed(secret)) if bit == "1"]
    lines += ["h q[0]", "h q[1]"]
    return HEADER + "qreg q[4];\ncreg c[2];\n" + "".join(f"{l};\n" for l in lines) + "measure q[0] -> c[0];\nmeasure q[1] -> c[1];\n"

for secret in ("11", "01"):
    counts = run("quantinuum_helios", simon(secret), shots=2000)["result"]["counts"]
    dots = {y: sum(int(a) * int(b) for a, b in zip(y, secret)) % 2 for y in counts}
    print(f"s = {secret}: measured y = {sorted(counts)}; y·s mod 2 = {dots}")

s = 11: measured y = ['00', '11']; y·s mod 2 = {'00': 0, '11': 0}
s = 01: measured y = ['00', '10']; y·s mod 2 = {'00': 0, '10': 0}


## Takeaways

- Every protocol works end to end on the twins, including mid-circuit measurement and feed-forward.
- Device noise lowers the CHSH value on each twin by a different amount, a quick, visible fidelity benchmark.